# 01c — Extract Weather (Open-Meteo Archive)
**Data source:** [Open-Meteo Archive API](https://open-meteo.com/en/docs/historical-weather-api)

**Features per lag window (7d, 14d, 30d):**
- `precip_sum_Xd` — total precipitation
- `precip_max_Xd` — max daily precipitation
- `temp_max_Xd`, `temp_min_Xd`, `temp_range_Xd` — temperature
- `wind_avg_Xd` — average wind speed

**Total:** 6 features x 3 lags = **18 weather features** per sample

**Input:** `train_base.parquet`, `val_base.parquet` from notebook 00

**Output:** `weather.parquet` (one row per unique station+date combo)

**Estimated time:** ~1-2 hours for ~9,500 samples (rate limited)

> Enable Internet. This is the LONGEST extraction — use Save & Run All.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, time, requests, logging
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-5s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('01c_weather')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titleweight': 'bold', 'font.size': 11})

INPUT_DIR  = '/kaggle/input/ey-water-quality-nb00'
OUTPUT_DIR = '/kaggle/working'
CKPT_DIR   = f'{OUTPUT_DIR}/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

LAT_COL     = 'Latitude'
LON_COL     = 'Longitude'
STATION_COL = 'station_id'
DATE_COL    = 'Sample Date'

LAGS = [7, 14, 30]
CHUNK_SIZE = 50  # save checkpoint every N rows

log.info(f'Lag windows: {LAGS}')

In [ ]:
train_base = pd.read_parquet(f'{INPUT_DIR}/train_base.parquet')
val_base   = pd.read_parquet(f'{INPUT_DIR}/val_base.parquet')

all_data = pd.concat([train_base, val_base], ignore_index=True)

# Unique (station, date) combos — weather is per sample, not per station
weather_keys = all_data[[STATION_COL, LAT_COL, LON_COL, DATE_COL]].drop_duplicates().reset_index(drop=True)

log.info(f'Train: {train_base.shape}, Val: {val_base.shape}')
log.info(f'Unique (station, date) combos: {len(weather_keys)}')

---
## Extraction with Checkpointing

Saves partial results every 50 rows so progress is not lost on timeout.

In [ ]:
def fetch_weather(lat, lon, date_str, lag=7, retries=3):
    """Fetch aggregated weather for a lag window before the sample date."""
    end_date = pd.to_datetime(date_str)
    start_date = end_date - timedelta(days=lag)
    
    for attempt in range(retries):
        try:
            r = requests.get('https://archive-api.open-meteo.com/v1/archive', params={
                'latitude': lat, 'longitude': lon,
                'start_date': start_date.strftime('%Y-%m-%d'),
                'end_date': end_date.strftime('%Y-%m-%d'),
                'daily': 'precipitation_sum,temperature_2m_max,temperature_2m_min,windspeed_10m_max',
                'timezone': 'Africa/Johannesburg'
            }, timeout=30)
            r.raise_for_status()
            d = r.json().get('daily', {})
            
            precip = [p for p in d.get('precipitation_sum', []) if p is not None]
            tmax   = [t for t in d.get('temperature_2m_max', []) if t is not None]
            tmin   = [t for t in d.get('temperature_2m_min', []) if t is not None]
            wind   = [w for w in d.get('windspeed_10m_max', []) if w is not None]
            
            return {
                f'precip_sum_{lag}d':   sum(precip) if precip else np.nan,
                f'precip_max_{lag}d':   max(precip) if precip else np.nan,
                f'temp_max_{lag}d':     max(tmax) if tmax else np.nan,
                f'temp_min_{lag}d':     min(tmin) if tmin else np.nan,
                f'temp_range_{lag}d':   (max(tmax) - min(tmin)) if tmax and tmin else np.nan,
                f'wind_avg_{lag}d':     np.mean(wind) if wind else np.nan,
            }
        except Exception:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    
    # All retries failed
    return {k: np.nan for k in [
        f'precip_sum_{lag}d', f'precip_max_{lag}d',
        f'temp_max_{lag}d', f'temp_min_{lag}d', f'temp_range_{lag}d',
        f'wind_avg_{lag}d'
    ]}

In [ ]:
# Check for existing checkpoint to resume from
ckpt_path = f'{CKPT_DIR}/weather_partial.parquet'
start_idx = 0
collected = []

if os.path.exists(ckpt_path):
    partial = pd.read_parquet(ckpt_path)
    start_idx = len(partial)
    collected = [partial]
    log.info(f'Resuming from checkpoint: row {start_idx}/{len(weather_keys)}')
else:
    log.info(f'Starting fresh extraction')

total = len(weather_keys)
log.info(f'Remaining: {total - start_idx} rows')

In [ ]:
start_time = time.time()
batch = []
failed = 0

for i in range(start_idx, total):
    row = weather_keys.iloc[i]
    lat, lon = row[LAT_COL], row[LON_COL]
    date_str = str(row[DATE_COL])
    station = row[STATION_COL]
    
    record = {
        STATION_COL: station,
        LAT_COL: lat, LON_COL: lon,
        DATE_COL: row[DATE_COL]
    }
    
    # Fetch all lag windows for this sample
    all_ok = True
    for lag in LAGS:
        wx = fetch_weather(lat, lon, date_str, lag)
        record.update(wx)
        if any(np.isnan(v) for v in wx.values()):
            all_ok = False
    
    batch.append(record)
    if not all_ok:
        failed += 1
    
    done = i + 1
    
    # Progress logging
    if done % 25 == 0 or done == total:
        elapsed = time.time() - start_time
        processed = done - start_idx
        rate = processed / elapsed if elapsed > 0 else 0
        remaining = total - done
        eta = remaining / rate if rate > 0 else 0
        log.info(f'  [{done:5d}/{total}] {done/total*100:5.1f}% | '
                 f'elapsed {elapsed/60:.1f}m | ETA {eta/60:.1f}m | '
                 f'rate {rate*60:.0f}/min | failed {failed}')
    
    # Checkpoint every CHUNK_SIZE rows
    if len(batch) >= CHUNK_SIZE:
        batch_df = pd.DataFrame(batch)
        all_so_far = pd.concat(collected + [batch_df], ignore_index=True)
        all_so_far.to_parquet(ckpt_path, index=False)
        collected = [all_so_far]
        batch = []
        log.info(f'  >> Checkpoint saved: {len(all_so_far)} rows')
    
    # Small delay to avoid rate limiting
    time.sleep(0.3)

# Flush remaining batch
if batch:
    batch_df = pd.DataFrame(batch)
    collected.append(batch_df)

elapsed_total = time.time() - start_time
log.info(f'DONE in {elapsed_total/60:.1f} min | {total} rows | {failed} partial failures')

In [ ]:
weather_df = pd.concat(collected, ignore_index=True)

log.info(f'Output shape: {weather_df.shape}')
log.info(f'Columns: {weather_df.columns.tolist()}')

# Per-column null summary
wx_cols = [c for c in weather_df.columns if c not in [STATION_COL, LAT_COL, LON_COL, DATE_COL]]
for col in wx_cols:
    nulls = weather_df[col].isnull().sum()
    pct = nulls / len(weather_df) * 100
    log.info(f'  {col:25s}: {nulls:4d} nulls ({pct:4.1f}%)')

display(weather_df[wx_cols].describe())

---
## Figure: Weather Feature Distributions

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

plot_cols = [c for c in wx_cols if '7d' in c][:6]  # show 7-day features
colors = ['#2196F3', '#2196F3', '#FF9800', '#FF9800', '#FF9800', '#4CAF50']

for i, col in enumerate(plot_cols):
    if i < len(axes):
        ax = axes[i]
        vals = weather_df[col].dropna()
        ax.hist(vals, bins=40, color=colors[i % len(colors)], alpha=0.7, edgecolor='white')
        ax.set_title(col, fontsize=10)
        ax.set_ylabel('Count')
        ax.axvline(vals.median(), color='red', linestyle='--', linewidth=1,
                   label=f'med={vals.median():.1f}')
        ax.legend(fontsize=8)

# Correlation across lag windows for precip
ax = axes[6]
lag_precip_cols = [f'precip_sum_{l}d' for l in LAGS if f'precip_sum_{l}d' in weather_df.columns]
if len(lag_precip_cols) >= 2:
    corr = weather_df[lag_precip_cols].corr()
    import seaborn as sns
    sns.heatmap(corr, annot=True, cmap='Blues', fmt='.2f', ax=ax, square=True)
    ax.set_title('Precip lag correlation', fontsize=10)

# Coverage summary
ax = axes[7]
null_pcts = [(c, weather_df[c].isnull().mean() * 100) for c in wx_cols]
null_pcts.sort(key=lambda x: x[1], reverse=True)
names = [n.replace('_', '\n') for n, _ in null_pcts[:8]]
vals = [v for _, v in null_pcts[:8]]
ax.barh(names, vals, color='coral')
ax.set_xlabel('% Missing')
ax.set_title('Missing values', fontsize=10)

for j in range(8, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Weather Features (7-day lag window)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_01c_weather_summary.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Save Output

In [ ]:
out_path = f'{OUTPUT_DIR}/weather.parquet'
weather_df.to_parquet(out_path, index=False)
size_mb = os.path.getsize(out_path) / (1024 * 1024)

# Clean up checkpoint
if os.path.exists(ckpt_path):
    os.remove(ckpt_path)
    log.info('Checkpoint cleaned up')

log.info(f'Saved: {out_path} ({size_mb:.1f} MB, {len(weather_df)} rows, {len(wx_cols)} features)')
print(f'\n=== DONE ===')
print(f'Output: weather.parquet')
print(f'Rows: {len(weather_df)}, Features: {len(wx_cols)}')
print(f'Next: add this notebook output as dataset input for 01e')